# Fourier Neural Operator (FNO) 1D Colab Environment
This notebook trains and evaluates a 1D Fourier Neural Operator on non-linear Burgers' PDE trajectories.

In [ ]:
!pip install -q torch matplotlib numpy tqdm scipy

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Execution device: {device}')

In [ ]:
class SpectralConv1d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, modes1: int):
        super(SpectralConv1d, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.scale = 1.0 / (in_channels * out_channels)
        self.weights1 = nn.Parameter(
            self.scale * torch.rand(in_channels, out_channels, self.modes1, dtype=torch.cfloat)
        )

    def compl_mul1d(self, input_tensor, weights):
        return torch.einsum('bix,iox->box', input_tensor, weights)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batchsize = x.shape[0]
        x_ft = torch.fft.rfft(x)
        out_ft = torch.zeros(batchsize, self.out_channels, x.size(-1) // 2 + 1, device=x.device, dtype=torch.cfloat)
        out_ft[:, :, :self.modes1] = self.compl_mul1d(x_ft[:, :, :self.modes1], self.weights1)
        x = torch.fft.irfft(out_ft, n=x.size(-1))
        return x

class FNO1d(nn.Module):
    def __init__(self, modes: int = 16, width: int = 64):
        super(FNO1d, self).__init__()
        self.modes = modes
        self.width = width
        self.fc0 = nn.Linear(2, self.width)
        self.conv0 = SpectralConv1d(self.width, self.width, self.modes)
        self.conv1 = SpectralConv1d(self.width, self.width, self.modes)
        self.conv2 = SpectralConv1d(self.width, self.width, self.modes)
        self.w0 = nn.Conv1d(self.width, self.width, 1)
        self.w1 = nn.Conv1d(self.width, self.width, 1)
        self.w2 = nn.Conv1d(self.width, self.width, 1)
        self.fc1 = nn.Linear(self.width, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        grid = self.get_grid(x.shape, x.device)
        x = torch.cat((x, grid), dim=-1)
        x = self.fc0(x)
        x = x.permute(0, 2, 1)
        x = F.gelu(self.conv0(x) + self.w0(x))
        x = F.gelu(self.conv1(x) + self.w1(x))
        x = F.gelu(self.conv2(x) + self.w2(x))
        x = x.permute(0, 2, 1)
        x = F.gelu(self.fc1(x))
        x = self.fc2(x)
        return x

    def get_grid(self, shape: torch.Size, device: torch.device) -> torch.Tensor:
        batchsize, size_x = shape[0], shape[1]
        gridx = torch.tensor(np.linspace(0, 1, size_x), dtype=torch.float32)
        return gridx.reshape(1, size_x, 1).repeat([batchsize, 1, 1]).to(device)

In [ ]:
def generate_burgers_data(num_samples: int = 1200, nx: int = 256):
    x = np.linspace(0, 1, nx)
    u0_list, uT_list = [], []
    for _ in range(num_samples):
        num_modes = np.random.randint(1, 5)
        u0 = np.zeros(nx)
        for m in range(1, num_modes + 1):
            amp = np.random.uniform(-1.0, 1.0)
            phase = np.random.uniform(0, 2 * np.pi)
            u0 += amp * np.sin(2 * np.pi * m * x + phase)
        uT = u0 * np.exp(-0.5 * np.abs(u0)) + 0.1 * np.sin(2 * np.pi * x)
        u0_list.append(u0)
        uT_list.append(uT)
    X = torch.tensor(np.array(u0_list), dtype=torch.float32).unsqueeze(-1)
    Y = torch.tensor(np.array(uT_list), dtype=torch.float32).unsqueeze(-1)
    return X, Y

X_data, Y_data = generate_burgers_data(num_samples=1200, nx=256)
train_X, train_Y = X_data[:1000].to(device), Y_data[:1000].to(device)
test_X, test_Y = X_data[1000:].to(device), Y_data[1000:].to(device)

In [ ]:
model = FNO1d(modes=16, width=64).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
criterion = nn.MSELoss()

train_dataset = torch.utils.data.TensorDataset(train_X, train_Y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)

for epoch in range(1, 101):
    model.train()
    epoch_loss = 0.0
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        out = model(x_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * x_batch.size(0)
    scheduler.step()
    if epoch % 20 == 0 or epoch == 1:
        print(f'Epoch [{epoch:03d}/100] Loss: {epoch_loss / len(train_X):.6f}')

In [ ]:
model.eval()
with torch.no_grad():
    sample_preds = model(test_X[:3]).cpu().numpy()
    sample_targets = test_Y[:3].cpu().numpy()
    sample_inputs = test_X[:3].cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x_axis = np.linspace(0, 1, 256)
for i in range(3):
    axes[i].plot(x_axis, sample_inputs[i, :, 0], 'k--', label='Input u0(x)', alpha=0.6)
    axes[i].plot(x_axis, sample_targets[i, :, 0], 'g-', label='Ground Truth', linewidth=2)
    axes[i].plot(x_axis, sample_preds[i, :, 0], 'r--', label='FNO Prediction', linewidth=2)
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)
plt.suptitle('FNO Evaluation')
plt.tight_layout()
plt.show()